# Neural Network
---

En este cuadernillo se realizará una comparación entre distintas arquitecturas de redes neuronales utilizando la librería `PyTorch`. Se evaluarán diferentes configuraciones para determinar cuál es la mejor arquitectura para el conjunto de datos utilizado.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [9]:
torch.manual_seed(42)
np.random.seed(42)

Tras instalar las librerías necesarias, se cargarán los datos de entrenamiento y validación, manteniendo el conjunto de prueba totalmente separado para una evaluación realista del modelo final.

In [ ]:
import sys
from pathlib import Path

root_path = Path.cwd().parent
sys.path.append(str(root_path))

from src.data import DataLoader

In [11]:
loader = DataLoader()
X_train, y_train, X_val, y_val, X_test, y_test = loader.load_data()

print(f"   Train: {X_train.shape[0]} samples")
print(f"   Val:   {X_val.shape[0]} samples")
print(f"   Test:  {X_test.shape[0]} samples")
print(f"   Classes: {loader.get_class_names()}")

   Train: 580800 samples
   Val:   11448 samples
   Test:  11472 samples
   Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']


A continuación, se definirán cinco arquitecturas de redes neuronales con diferentes niveles de complejidad.

La capa final de cada arquitectura no incluye una función de activación `Softmax`, ya que se utilizará la función de pérdida `CrossEntropyLoss` y esta función de pérdida ya aplica `Softmax` internamente. Por lo tanto, no es necesario agregar una capa de activación adicional al final de la red puesto que podría causar problemas de convergencia durante el entrenamiento al aplicar `Softmax` dos veces.

In [12]:
# Arquitectura 1 - Simple MLP
class SimpleMLP(nn.Module):
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(77, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 24)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out
    
# Arquitectura 2 - MLP con 2 capas ocultas
class Complex2MLP(nn.Module):
    def __init__(self):
        super(Complex2MLP, self).__init__()
        self.fc1 = nn.Linear(77, 256)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(128, 24)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        return out
    
# Arquitectura 3 - MLP con 3 capas ocultas
class Complex3MLP(nn.Module):
    def __init__(self):
        super(Complex3MLP, self).__init__()
        self.fc1 = nn.Linear(77, 256)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(128, 64)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(64, 24)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        out = self.relu3(out)
        out = self.fc4(out)
        return out
    
# Arquitectura 4 - MLP con 4 capas ocultas
class Complex4MLP(nn.Module):
    def __init__(self):
        super(Complex4MLP, self).__init__()
        self.fc1 = nn.Linear(77, 256)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(128, 128)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(128, 64)
        self.relu4 = nn.ReLU()
        self.fc5 = nn.Linear(64, 24)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        out = self.relu3(out)
        out = self.fc4(out)
        out = self.relu4(out)
        out = self.fc5(out)
        return out
    
# Arquitectura 5 - MLP con 6 capas ocultas
class Complex6MLP(nn.Module):
    def __init__(self):
        super(Complex6MLP, self).__init__()
        self.fc1 = nn.Linear(77, 256)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(128, 128)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(128, 64)
        self.relu4 = nn.ReLU()
        self.fc5 = nn.Linear(64, 64)
        self.relu5 = nn.ReLU()
        self.fc6 = nn.Linear(64, 24)
        self.relu6 = nn.ReLU()
        self.fc7 = nn.Linear(24, 24)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        out = self.relu3(out)
        out = self.fc4(out)
        out = self.relu4(out)
        out = self.fc5(out)
        out = self.relu5(out)
        out = self.fc6(out)
        out = self.relu6(out)
        out = self.fc7(out)
        return out


In [14]:
# Train and evaluate the models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def train_model(model, X_train, y_train, X_val, y_val, num_epochs=500, learning_rate=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(num_epochs):
        model.train()
        inputs = torch.tensor(X_train, dtype=torch.float32, device=device)
        labels = torch.tensor(y_train, dtype=torch.long, device=device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluate on validation set
    model.eval()
    with torch.no_grad():
        val_inputs = torch.tensor(X_val, dtype=torch.float32, device=device)
        val_outputs = model(val_inputs)
        _, predicted = torch.max(val_outputs.data, 1)

    predicted_np = predicted.detach().cpu().numpy()
    accuracy = accuracy_score(y_val, predicted_np)
    f1 = f1_score(y_val, predicted_np, average='macro')
    
    return accuracy, f1

input_size = X_train.shape[1]
num_classes = len(loader.get_class_names())

# Simple MLP
print("Training Simple MLP...")
simple_mlp = SimpleMLP().to(device)
simple_accuracy, simple_f1 = train_model(simple_mlp, X_train, y_train, X_val, y_val)

# Complex MLP con 2 capas ocultas
print("Training Complex MLP with 2 hidden layers...")
complex2_mlp = Complex2MLP().to(device)
complex2_accuracy, complex2_f1 = train_model(complex2_mlp, X_train, y_train, X_val, y_val)

# Complex MLP con 3 capas ocultas
print("Training Complex MLP with 3 hidden layers...")
complex3_mlp = Complex3MLP().to(device)
complex3_accuracy, complex3_f1 = train_model(complex3_mlp, X_train, y_train, X_val, y_val)

# Complex MLP con 4 capas ocultas
print("Training Complex MLP with 4 hidden layers...")
complex4_mlp = Complex4MLP().to(device)
complex4_accuracy, complex4_f1 = train_model(complex4_mlp, X_train, y_train, X_val, y_val)

# Complex MLP con 6 capas ocultas
print("Training Complex MLP with 6 hidden layers...")
complex6_mlp = Complex6MLP().to(device)
complex6_accuracy, complex6_f1 = train_model(complex6_mlp, X_train, y_train, X_val, y_val)

print("\nPerformance Comparison:")
print("Simple MLP - Accuracy: {:.4f}, F1 Score: {:.4f}".format(simple_accuracy, simple_f1))
print("Complex MLP (2 hidden layers) - Accuracy: {:.4f}, F1 Score: {:.4f}".format(complex2_accuracy, complex2_f1))
print("Complex MLP (3 hidden layers) - Accuracy: {:.4f}, F1 Score: {:.4f}".format(complex3_accuracy, complex3_f1))
print("Complex MLP (4 hidden layers) - Accuracy: {:.4f}, F1 Score: {:.4f}".format(complex4_accuracy, complex4_f1))
print("Complex MLP (6 hidden layers) - Accuracy: {:.4f}, F1 Score: {:.4f}".format(complex6_accuracy, complex6_f1))

Using device: cuda
Training Simple MLP...
Training Complex MLP with 2 hidden layers...
Training Complex MLP with 3 hidden layers...
Training Complex MLP with 4 hidden layers...
Training Complex MLP with 6 hidden layers...

Performance Comparison:
Simple MLP - Accuracy: 0.9113, F1 Score: 0.9114
Complex MLP (2 hidden layers) - Accuracy: 0.9334, F1 Score: 0.9334
Complex MLP (3 hidden layers) - Accuracy: 0.9317, F1 Score: 0.9317
Complex MLP (4 hidden layers) - Accuracy: 0.9095, F1 Score: 0.9091
Complex MLP (6 hidden layers) - Accuracy: 0.8881, F1 Score: 0.8882


In [15]:
# Mostrar clasificación report y matriz de confusión para el mejor modelo
best_model = max([(simple_mlp, simple_accuracy),
                    (complex2_mlp, complex2_accuracy),
                    (complex3_mlp, complex3_accuracy),
                    (complex4_mlp, complex4_accuracy),
                    (complex6_mlp, complex6_accuracy)], key=lambda x: x[1])[0]
best_model.eval()
with torch.no_grad():
    test_inputs = torch.tensor(X_test, dtype=torch.float32, device=device)
    test_outputs = best_model(test_inputs)
    _, predicted = torch.max(test_outputs.data, 1)

predicted_np = predicted.detach().cpu().numpy()

print("\nBest Model:", best_model.__class__.__name__)
print("Classification Report:")
print(classification_report(y_test, predicted_np, target_names=loader.get_class_names()))
print("Confusion Matrix:")
print(confusion_matrix(y_test, predicted_np))


Best Model: Complex2MLP
Classification Report:
              precision    recall  f1-score   support

           A       0.90      0.95      0.92       478
           B       0.98      0.99      0.98       478
           C       0.90      0.93      0.92       478
           D       0.92      0.87      0.89       478
           E       0.94      0.93      0.94       478
           F       0.92      0.98      0.95       478
           G       0.95      0.95      0.95       478
           H       0.94      0.90      0.92       478
           I       0.98      0.95      0.96       478
           K       0.92      0.97      0.94       478
           L       0.97      0.98      0.97       478
           M       0.75      0.90      0.82       478
           N       0.93      0.77      0.84       478
           O       0.90      0.95      0.92       478
           P       0.97      0.95      0.96       478
           Q       0.94      0.92      0.93       478
           R       0.93      0.90

Tal y como se puede observar, la complejidad de la arquitectura no siempre se traduce en un mejor rendimiento. El mejor modelo entrenado consiste en el `Complex MLP con 2 capas ocultas`, aunque está muy cerca del `Complex MLP con 3 capas ocultas`.